# 1. Exploratory Data Analysis

In [15]:
import pandas as pd

# Läs in rådata från Excel-filen
df_raw = pd.read_excel("../data/WHR26_Data_Figure_2.1.xlsx")

# Kontrollera antal rader och kolumner
print(df_raw.shape)

# Visa datatyper för varje kolumn
print(df_raw.dtypes)

# Visa de 5 första raderna för att förstå strukturen
df_raw.head()

(2116, 13)
Year                                            int64
Rank                                            int64
Country name                                      str
Life evaluation (3-year average)              float64
Lower whisker                                 float64
Upper whisker                                 float64
Explained by: Log GDP per capita              float64
Explained by: Social support                  float64
Explained by: Healthy life expectancy         float64
Explained by: Freedom to make life choices    float64
Explained by: Generosity                      float64
Explained by: Perceptions of corruption       float64
Dystopia + residual                           float64
dtype: object


,Year,Rank,Country name,Life evaluation (3-year average),Lower whisker,Upper whisker,Explained by: Log GDP per capita,Explained by: Social support,Explained by: Healthy life expectancy,Explained by: Freedom to make life choices,Explained by: Generosity,Explained by: Perceptions of corruption,Dystopia + residual
0,2025,1,Finland,7.764,7.690,7.837,1.915,1.638,0.939,1.105,0.093,0.491,1.582
1,2025,2,Iceland,7.540,7.449,7.630,1.971,1.720,0.996,1.105,0.187,0.187,1.373
2,2025,3,Denmark,7.539,7.446,7.631,1.986,1.633,0.930,1.081,0.125,0.474,1.310
3,2025,4,Costa Rica,7.439,7.356,7.522,1.697,1.483,0.739,1.101,0.059,0.122,2.236
4,2025,5,Sweden,7.255,7.172,7.337,1.950,1.570,1.027,1.070,0.149,0.447,1.041


In [16]:
# Välj ut de kolumner vi behöver för analysen
kolumner = [
    "Year",
    "Country name",
    "Life evaluation (3-year average)",
    "Explained by: Log GDP per capita",
    "Explained by: Social support",
    "Explained by: Healthy life expectancy",
    "Explained by: Freedom to make life choices",
    "Explained by: Generosity",
    "Explained by: Perceptions of corruption"
]

# Filtrera till åren 2019-2025
df = df_raw[df_raw["Year"].between(2019, 2025)][kolumner].copy()

# Kontrollera resultatet
print(df.shape)
df.head()

(1022, 9)


,Year,Country name,Life evaluation (3-year average),Explained by: Log GDP per capita,Explained by: Social support,Explained by: Healthy life expectancy,Explained by: Freedom to make life choices,Explained by: Generosity,Explained by: Perceptions of corruption
0,2025,Finland,7.764,1.915,1.638,0.939,1.105,0.093,0.491
1,2025,Iceland,7.540,1.971,1.720,0.996,1.105,0.187,0.187
2,2025,Denmark,7.539,1.986,1.633,0.930,1.081,0.125,0.474
3,2025,Costa Rica,7.439,1.697,1.483,0.739,1.101,0.059,0.122
4,2025,Sweden,7.255,1.950,1.570,1.027,1.070,0.149,0.447


In [17]:
# Kontrollera hur många saknade värden som finns per kolumn
print(df.isnull().sum())

# Droppa rader där någon av de 6 lyckofaktorerna saknas
df = df.dropna()

# Bekräfta att inga NaN finns kvar och hur många rader som är kvar
print(f"\nEfter borttagning av NaN: {df.shape[0]} rader, {df.shape[1]} kolumner")

Year                                          0
Country name                                  0
Life evaluation (3-year average)              0
Explained by: Log GDP per capita              3
Explained by: Social support                  3
Explained by: Healthy life expectancy         6
Explained by: Freedom to make life choices    5
Explained by: Generosity                      3
Explained by: Perceptions of corruption       4
dtype: int64

Efter borttagning av NaN: 1013 rader, 9 kolumner


Vi fokuserar på de 6 lyckofaktorerna som WHR använder för att förklara lyckopoängen:

- **BNP** – Log GDP per capita
- **Socialt stöd** – Social support
- **Hälsa** – Healthy life expectancy
- **Frihet** – Freedom to make life choices
- **Generositet** – Generosity
- **Korruption** – Perceptions of corruption

Utfallsvariabel: **Life evaluation (3-year average)**
Analysenhet: 143 länder, åren 2019–2025

In [18]:
# Statistisk översikt: visar min, max, medelvärde och standardavvikelse för varje numerisk kolumn — hjälper oss se om skalorna skiljer sig åt och om några värden verkar orimliga
df.describe().round(3)

,Year,Life evaluation (3-year average),Explained by: Log GDP per capita,Explained by: Social support,Explained by: Healthy life expectancy,Explained by: Freedom to make life choices,Explained by: Generosity,Explained by: Perceptions of corruption
count,1013.000,1013.000,1013.000,1013.000,1013.000,1013.000,1013.000,1013.000
mean,2021.961,5.553,1.266,1.095,0.553,0.609,0.148,0.145
std,2.022,1.127,0.465,0.358,0.230,0.212,0.084,0.119
min,2019.000,1.364,0.000,0.000,0.000,0.000,0.000,0.000
25%,2020.000,4.729,0.944,0.865,0.389,0.471,0.089,0.063
50%,2022.000,5.714,1.306,1.135,0.560,0.602,0.134,0.113
75%,2024.000,6.387,1.636,1.379,0.712,0.735,0.196,0.181
max,2025.000,7.842,2.209,1.840,1.238,1.147,0.570,0.587


In [19]:
# Kontrollera hur många länder som rapporterat data per år
# Om ett år har färre länder kan det påverka vår jämförelse
print(df.groupby("Year")["Country name"].count())

# Säkerställ att vi har jämförbara länder i både 2019 och 2025 — grunden för vinnare/förlorare-analysen
länder_2019 = set(df[df["Year"] == 2019]["Country name"])
länder_2025 = set(df[df["Year"] == 2025]["Country name"])
länder_båda = länder_2019 & länder_2025

# Visa hur många länder som finns i varje år och i båda åren
print(f"\nLänder i 2019: {len(länder_2019)}")
print(f"Länder i 2025: {len(länder_2025)}")
print(f"Länder i BÅDA åren: {len(länder_båda)}")

Year
2019    153
2020    149
2021    146
2022    136
2023    140
2024    144
2025    145
Name: Country name, dtype: int64

Länder i 2019: 153
Länder i 2025: 145
Länder i BÅDA åren: 141


In [20]:
# Filtrera datan till endast de 141 länder som finns i BÅDA 2019 och 2025
# Dessa är de enda länder vi kan jämföra för vinnare/förlorare-analysen
df = df[df["Country name"].isin(länder_båda)].copy()

# Bekräfta att vi nu har 141 länder och inga saknade värden
print(f"Antal rader efter filtrering: {df.shape[0]}")
print(f"Antal unika länder: {df['Country name'].nunique()}")

Antal rader efter filtrering: 975
Antal unika länder: 141


In [27]:
# Exportera det renade datasetet till CSV
# Kan öppnas direkt i Google Sheets eller Excel
df.to_csv("../data/cleaned_data.csv", index=False)
print("Exporterat till data/cleaned_data.csv")

Exporterat till data/cleaned_data.csv
